In [32]:
import os
import sys
import tempfile
from pathlib import Path
from time import time

import cv2
import numpy as np
import torch
import trimesh

In [33]:
# 路径设置
THIS_FILE_DIR = Path('/home/viplab/liukejia/project/reconstruct').resolve()
PROJECT_ROOT = THIS_FILE_DIR.parent
VGGT_ROOT = PROJECT_ROOT / 'vggt'

if str(THIS_FILE_DIR) not in sys.path:
    sys.path.append(str(THIS_FILE_DIR))
if str(VGGT_ROOT) not in sys.path:
    sys.path.append(str(VGGT_ROOT))
if str(VGGT_ROOT / 'vggt') not in sys.path:
    sys.path.append(str(VGGT_ROOT / 'vggt'))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8) else torch.float16
print('device:', device, 'dtype:', dtype)

device: cuda dtype: torch.bfloat16


In [ ]:
# 参数区（按需修改）
folder_names = [
    '/data/test/test_30/2025-10-22_15-49-02',
    '/data/test/test_30/2025-10-22_15-49-03',
    '/data/test/test_30/2025-10-22_15-49-04',
    '/data/test/test_30/2025-10-22_15-49-05',
]

# 为了和 baseline 可比，先用同一类输入（单张 4-in-1 图）
source_image_path = '/home/viplab/liukejia/project/mv_model/results_4in1_sdxl_warp_ckpt13000/result_2001.png'

output_dir = Path('/home/viplab/liukejia/project/reconstruct/reconstruct_glb')
# single_512_to_glb 默认 50，更稳健
conf_thres = 50.0

assert len(folder_names) == 4, 'folder_names 必须是 4 个视角文件夹'
for f in folder_names:
    if not Path(f).exists():
        raise FileNotFoundError(f'Folder not found: {f}')
if not Path(source_image_path).exists():
    raise FileNotFoundError(f'Source image not found: {source_image_path}')

print('source_image_path:', source_image_path)
print('output_dir:', output_dir)
print('conf_thres:', conf_thres)

source_image_path: /home/viplab/liukejia/project/mv_model/results_grid
output_dir: /home/viplab/liukejia/project/reconstruct/results_grid_glb
conf_thres: 50.0


In [35]:
from utils.camera import parse_json_camera_extrinsic_intrinsic
from visual_util import predictions_to_glb
from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images
from vggt.utils.pose_enc import pose_encoding_to_extri_intri
from vggt.utils.geometry import unproject_depth_map_to_point_map

In [36]:
def read_single_image_512(image_path: str, force_resize: bool = True) -> np.ndarray:
    bgr = cv2.imread(image_path, cv2.IMREAD_COLOR)
    if bgr is None:
        raise ValueError(f'Failed to read image: {image_path}')

    h, w = bgr.shape[:2]
    if (h, w) != (512, 512):
        if force_resize:
            bgr = cv2.resize(bgr, (512, 512), interpolation=cv2.INTER_AREA)
            print(f'[Info] resized {image_path}: {(w, h)} -> (512, 512)')
        else:
            raise ValueError(f'Expected 512x512, got {(w, h)}')

    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


def split_512_to_4x256_and_save(img_rgb: np.ndarray, target_dir: Path):
    images_dir = target_dir / 'images'
    images_dir.mkdir(parents=True, exist_ok=True)

    patches = [
        img_rgb[0:256, 0:256],
        img_rgb[0:256, 256:512],
        img_rgb[256:512, 0:256],
        img_rgb[256:512, 256:512],
    ]

    patch_paths = []
    for i, p in enumerate(patches):
        out_path = images_dir / f'{i:06d}.png'
        cv2.imwrite(str(out_path), cv2.cvtColor(p, cv2.COLOR_RGB2BGR))
        patch_paths.append(str(out_path))

    return patches, patch_paths

In [37]:
# 1) 单图一切四
img_rgb = read_single_image_512(source_image_path, force_resize=True)
tmp_dir = Path(tempfile.mkdtemp(prefix='vggt_depth_split_'))
patches, patch_image_paths = split_512_to_4x256_and_save(img_rgb, tmp_dir)
print('split patch dir:', tmp_dir / 'images')

predictions = {}
predictions['images'] = np.stack(patches, axis=0).astype(np.float32) / 255.0  # (4, 256, 256, 3)

ValueError: Failed to read image: /home/viplab/liukejia/project/mv_model/results_grid

In [ ]:
# 2) 先准备文件相机路径（稍后按 VGGT 输入分辨率解析）
json_paths = [f'{folder}/camconf.json' for folder in folder_names]
print('json_paths:')
for p in json_paths:
    print('-', p)

# 是否在推理后用文件相机覆盖 VGGT 相机
use_file_camera = True

json_paths:
- /data/test/test_30/2025-10-22_15-49-02/camconf.json
- /data/test/test_30/2025-10-22_15-49-03/camconf.json
- /data/test/test_30/2025-10-22_15-49-04/camconf.json
- /data/test/test_30/2025-10-22_15-49-05/camconf.json


In [ ]:
# 3) 尽量按 single_512_to_glb 的链路：VGGT 输出 + pose 解码
print('Initializing VGGT...')
model = VGGT()
_URL = 'https://huggingface.co/facebook/VGGT-1B/resolve/main/model.pt'
state_dict = torch.hub.load_state_dict_from_url(_URL, map_location='cpu')
model.load_state_dict(state_dict)
model.eval().to(device)

images_for_model = load_and_preprocess_images(patch_image_paths).to(device)
H, W = int(images_for_model.shape[-2]), int(images_for_model.shape[-1])
print('model input H,W =', H, W)

with torch.no_grad():
    if torch.cuda.is_available():
        with torch.cuda.amp.autocast(dtype=dtype):
            model_outputs = model(images_for_model)
    else:
        model_outputs = model(images_for_model)

# 与 single_512_to_glb 一致：由 pose_enc 解码相机
extri_pred, intri_pred = pose_encoding_to_extri_intri(model_outputs['pose_enc'], (H, W))
model_outputs['extrinsic'] = extri_pred
model_outputs['intrinsic'] = intri_pred

# Tensor -> numpy（去 batch 维）
predictions = {}
for k, v in model_outputs.items():
    if isinstance(v, torch.Tensor):
        predictions[k] = v.detach().cpu().numpy().squeeze(0)
    else:
        predictions[k] = v

predictions['pose_enc_list'] = None

# 可选：用文件相机做刚体对齐（旋转+平移），让第一个相机与 json 第一个相机完全重合
if use_file_camera:
    def se3_to_4x4_batch(mats: np.ndarray) -> np.ndarray:
        mats = np.asarray(mats, dtype=np.float32)
        if mats.ndim != 3:
            raise ValueError(f'se3 维度应为 (N,3,4) 或 (N,4,4)，实际: {mats.shape}')
        if mats.shape[1:] == (4, 4):
            return mats
        if mats.shape[1:] == (3, 4):
            out = np.tile(np.eye(4, dtype=np.float32), (mats.shape[0], 1, 1))
            out[:, :3, :4] = mats
            return out
        raise ValueError(f'不支持的 se3 形状: {mats.shape}')

    file_extr_list = []
    for jp in json_paths:
        ext44, _ = parse_json_camera_extrinsic_intrinsic(
            jp, device='cpu', H=H, W=W, dtype=torch.float32, flip_y=True
        )
        file_extr_list.append(ext44.squeeze(0).detach().cpu().numpy())

    pred_extr_raw = predictions['extrinsic'].astype(np.float32).copy()
    if len(file_extr_list) != pred_extr_raw.shape[0]:
        raise ValueError(f'json_paths 数量({len(file_extr_list)})与相机数量({pred_extr_raw.shape[0]})不一致')
    file_extr_raw = np.stack(file_extr_list, axis=0).astype(np.float32)

    # 兼容 (N,3,4) 与 (N,4,4)
    pred_extr4 = se3_to_4x4_batch(pred_extr_raw)
    file_extr4 = se3_to_4x4_batch(file_extr_raw)

    # world2cam -> c2w
    pred_c2w = np.linalg.inv(pred_extr4)
    file_c2w = np.linalg.inv(file_extr4)

    # 取第一个相机，求全局刚体变换 G=[Q,d]，满足：
    # R_file0 = Q @ R_pred0,  C_file0 = Q @ C_pred0 + d
    R_pred0 = pred_c2w[0, :3, :3]
    C_pred0 = pred_c2w[0, :3, 3]
    R_file0 = file_c2w[0, :3, :3]
    C_file0 = file_c2w[0, :3, 3]

    Q = R_file0 @ R_pred0.T
    d = C_file0 - Q @ C_pred0

    # 把同一刚体变换应用到所有预测相机
    pred_c2w_new = pred_c2w.copy()
    pred_c2w_new[:, :3, :3] = np.einsum('ij,njk->nik', Q, pred_c2w[:, :3, :3])
    pred_c2w_new[:, :3, 3] = np.einsum('ij,nj->ni', Q, pred_c2w[:, :3, 3]) + d

    # 转回 world2cam，并保持原始外参形状
    pred_extr_new4 = np.linalg.inv(pred_c2w_new).astype(np.float32)
    if pred_extr_raw.shape[1:] == (3, 4):
        predictions['extrinsic'] = pred_extr_new4[:, :3, :4]
    elif pred_extr_raw.shape[1:] == (4, 4):
        predictions['extrinsic'] = pred_extr_new4
    else:
        raise ValueError(f'predictions["extrinsic"] 形状异常: {pred_extr_raw.shape}')

    # 对齐检查（第一个相机）
    R0_new = pred_c2w_new[0, :3, :3]
    C0_new = pred_c2w_new[0, :3, 3]
    rot_err = np.linalg.norm(R0_new - R_file0)
    ctr_err = np.linalg.norm(C0_new - C_file0)
    print('[use_file_camera] applied rigid transform (rotation + translation)')
    print('[use_file_camera] center error of cam0:', ctr_err)
    print('[use_file_camera] rotation matrix error of cam0:', rot_err)

# 确保 conf 存在
if 'depth_conf' not in predictions:
    predictions['depth_conf'] = np.ones_like(predictions['depth'][..., 0], dtype=np.float32)

# 如果模型没给 images，则使用 resize 后的 RGB
if 'images' not in predictions:
    resized_rgb = []
    for p in patches:
        p_rs = cv2.resize(p, (W, H), interpolation=cv2.INTER_LINEAR)
        resized_rgb.append(p_rs.astype(np.float32) / 255.0)
    predictions['images'] = np.stack(resized_rgb, axis=0)

print('images shape:', predictions['images'].shape)
print('depth shape:', predictions['depth'].shape)
print('extrinsic shape:', predictions['extrinsic'].shape)
print('intrinsic shape:', predictions['intrinsic'].shape)
print('depth min/max:', np.nanmin(predictions['depth']), np.nanmax(predictions['depth']))

Initializing VGGT...
model input H,W = 518 518
[use_file_camera] applied rigid transform (rotation + translation)
[use_file_camera] center error of cam0: 0.0
[use_file_camera] rotation matrix error of cam0: 8.940697e-08
images shape: (4, 3, 518, 518)
depth shape: (4, 518, 518, 1)
extrinsic shape: (4, 3, 4)
intrinsic shape: (4, 3, 3)
depth min/max: 0.7029413 1.8310616


In [ ]:
# （可选检查）depth/conf 统计
print('depth shape:', predictions['depth'].shape)
print('depth_conf shape:', predictions['depth_conf'].shape)
print('depth_conf min/max:', np.nanmin(predictions['depth_conf']), np.nanmax(predictions['depth_conf']))

depth shape: (4, 518, 518, 1)
depth_conf shape: (4, 518, 518)
depth_conf min/max: 1.0000002 9.003549


In [ ]:
# （可选检查）当前导出输入统计
print('images range:', float(np.nanmin(predictions['images'])), float(np.nanmax(predictions['images'])))
print('depth range:', float(np.nanmin(predictions['depth'])), float(np.nanmax(predictions['depth'])))
print('use_file_camera:', use_file_camera)

images range: 0.0 1.0
depth range: 0.7029412984848022 1.831061601638794
use_file_camera: True


In [ ]:
# 4) 反投影 + 导出 GLB（与 baseline 一致，不做额外坐标拟合）
world_points = unproject_depth_map_to_point_map(
    predictions['depth'],
    predictions['extrinsic'],
    predictions['intrinsic'],
)
predictions['world_points_from_depth'] = world_points

output_dir.mkdir(parents=True, exist_ok=True)
glbfile = output_dir / f'{time()}.glb'

print(f'Exporting to {glbfile} ...')
glbscene = predictions_to_glb(
    predictions,
    conf_thres=conf_thres,
    filter_by_frames='All',
    mask_black_bg=False,
    mask_white_bg=False,
    show_cam=True,
    mask_sky=False,
    target_dir=str(output_dir),
    prediction_mode='Depthmap and Camera Branch',
)
glbscene.export(file_obj=str(glbfile))
print('Done.')
print('GLB:', glbfile)

Exporting to /home/viplab/liukejia/project/reconstruct/reconstruct_glb/1776069329.5041637.glb ...
Building GLB scene
Using Depthmap and Camera Branch
GLB Scene built
Done.
GLB: /home/viplab/liukejia/project/reconstruct/reconstruct_glb/1776069329.5041637.glb
